# Task 3 — Model Deployment: FastAPI + Docker
**Alfido Tech Internship**  
Goal: Wrap the trained churn model (from Task 1) in a FastAPI REST API and containerize with Docker.

> **Prerequisite:** Run Task 1 notebook first. It saves `model.pkl`, `scaler.pkl`, and `feature_names.pkl` which this task uses.

## Cell 1 — Verify Model Artifacts Exist

In [ ]:
import os, joblib, numpy as np

# These files are generated by Task 1 notebook
required = ['../model.pkl', '../scaler.pkl', '../feature_names.pkl']

for f in required:
    exists = os.path.exists(f)
    print(f"{'OK' if exists else 'MISSING'}  {f}")

model         = joblib.load('../model.pkl')
scaler        = joblib.load('../scaler.pkl')
feature_names = joblib.load('../feature_names.pkl')

print(f'\nModel type: {type(model).__name__}')
print(f'Features:   {len(feature_names)}')
print(f'Feature list: {feature_names}')

## Cell 2 — Test the Model Locally (Before Wrapping in API)

In [ ]:
# Simulate a high-risk customer (month-to-month, fiber, 8 months tenure)
sample = {
    'gender': 1, 'SeniorCitizen': 0, 'Partner': 0, 'Dependents': 0,
    'tenure': 8, 'PhoneService': 1, 'MultipleLines': 0,
    'InternetService': 1, 'OnlineSecurity': 0, 'OnlineBackup': 0,
    'DeviceProtection': 0, 'TechSupport': 0, 'StreamingTV': 0,
    'StreamingMovies': 0, 'Contract': 0, 'PaperlessBilling': 1,
    'PaymentMethod': 2, 'MonthlyCharges': 72.5, 'TotalCharges': 580.0
}

x = np.array([[sample[f] for f in feature_names]])
x_scaled = scaler.transform(x)

prob  = model.predict_proba(x_scaled)[0, 1]
churn = prob >= 0.5

print(f'Churn probability: {prob:.4f}')
print(f'Prediction: {"CHURN" if churn else "NO CHURN"}')
print(f'Risk level: {"HIGH" if prob>=0.7 else "MEDIUM" if prob>=0.45 else "LOW"}')

## Cell 3 — Install FastAPI and Uvicorn

In [ ]:
# Run this once to install FastAPI
import subprocess
result = subprocess.run(
    ['pip', 'install', 'fastapi==0.111.0', 'uvicorn==0.30.1', 'pydantic==2.7.4'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else 'Already installed')
print('FastAPI ready.')

## Cell 4 — Show the Full API Code (app/main.py)

In [ ]:
with open('../app/main.py', 'r') as f:
    print(f.read())

## Cell 5 — Start the API Server

In [ ]:
# Run this in a TERMINAL (not the notebook) — it starts the live server:
#
#   cd task3-deployment
#   uvicorn app.main:app --reload
#
# Then open http://localhost:8000/docs in your browser.
# This cell just prints the command as a reminder.

print('To start the server, run in terminal:')
print()
print('    cd task3-deployment')
print('    uvicorn app.main:app --reload')
print()
print('Then open: http://localhost:8000/docs')

## Cell 6 — Test API with Python requests (server must be running)

In [ ]:
# Make sure uvicorn is running in a terminal before running this cell
import requests, json

BASE_URL = 'http://localhost:8000'

# ── Health check ──
health = requests.get(f'{BASE_URL}/health').json()
print('Health check:', json.dumps(health, indent=2))

# ── High-risk customer ──
payload = {
    'gender': 1, 'SeniorCitizen': 0, 'Partner': 0, 'Dependents': 0,
    'tenure': 8, 'PhoneService': 1, 'MultipleLines': 0,
    'InternetService': 1, 'OnlineSecurity': 0, 'OnlineBackup': 0,
    'DeviceProtection': 0, 'TechSupport': 0, 'StreamingTV': 0,
    'StreamingMovies': 0, 'Contract': 0, 'PaperlessBilling': 1,
    'PaymentMethod': 2, 'MonthlyCharges': 72.5, 'TotalCharges': 580.0
}

response = requests.post(f'{BASE_URL}/predict', json=payload)
print('\nHigh-risk customer prediction:')
print(json.dumps(response.json(), indent=2))

# ── Low-risk customer ──
payload_low = payload.copy()
payload_low.update({'tenure': 48, 'Contract': 2, 'MonthlyCharges': 45.0, 'TotalCharges': 2160.0})

response_low = requests.post(f'{BASE_URL}/predict', json=payload_low)
print('\nLow-risk customer prediction:')
print(json.dumps(response_low.json(), indent=2))

## Cell 7 — Equivalent curl Commands (for README demo)

In [ ]:
curl_health = 'curl http://localhost:8000/health'

curl_predict = '''curl -X POST http://localhost:8000/predict \\
  -H "Content-Type: application/json" \\
  -d @sample_request.json'''

print('Health check')
print(curl_health)
print()
print('Predict (using sample_request.json)')
print(curl_predict)
print()
print('Expected response:')
expected = {
    'churn': True,
    'probability': 0.7312,
    'confidence': 'high',
    'risk_level': 'HIGH RISK',
    'recommendation': 'Immediate retention action recommended — offer a contract upgrade or discount.'
}
import json
print(json.dumps(expected, indent=2))

## Cell 8 — Show Dockerfile

In [ ]:
with open('../Dockerfile', 'r') as f:
    print(f.read())

## Cell 9 — Docker Build & Run Commands

In [ ]:
docker_commands = """
# Step 1 — Build the Docker image
docker build -t churn-api .

# Step 2 — Run the container (maps port 8000)
docker run -d -p 8000:8000 --name churn-api churn-api

# Step 3 — Check it's running
docker ps
curl http://localhost:8000/health

# Step 4 — Send a prediction
curl -X POST http://localhost:8000/predict \\
  -H \"Content-Type: application/json\" \\
  -d @sample_request.json

# Stop the container
docker stop churn-api && docker rm churn-api
"""
print(docker_commands)

## Cell 10 — Test Multiple Customers & Show Results Table

In [ ]:
import pandas as pd

# Simulate 5 different customer profiles
profiles = [
    {'label': 'High-risk (month-to-month, fiber, new)',
     'tenure': 4,  'Contract': 0, 'MonthlyCharges': 80.0, 'TotalCharges': 320.0,
     'InternetService': 1, 'OnlineSecurity': 0, 'TechSupport': 0},
    {'label': 'Medium-risk (month-to-month, DSL, 1yr)',
     'tenure': 14, 'Contract': 0, 'MonthlyCharges': 55.0, 'TotalCharges': 770.0,
     'InternetService': 0, 'OnlineSecurity': 0, 'TechSupport': 0},
    {'label': 'Low-risk (2yr contract, long tenure)',
     'tenure': 60, 'Contract': 2, 'MonthlyCharges': 45.0, 'TotalCharges': 2700.0,
     'InternetService': 0, 'OnlineSecurity': 1, 'TechSupport': 1},
    {'label': 'Low-risk (1yr contract, 3 years)',
     'tenure': 36, 'Contract': 1, 'MonthlyCharges': 50.0, 'TotalCharges': 1800.0,
     'InternetService': 0, 'OnlineSecurity': 1, 'TechSupport': 0},
    {'label': 'Very high-risk (new, fiber, no support)',
     'tenure': 2,  'Contract': 0, 'MonthlyCharges': 95.0, 'TotalCharges': 190.0,
     'InternetService': 1, 'OnlineSecurity': 0, 'TechSupport': 0},
]

defaults = {
    'gender': 1, 'SeniorCitizen': 0, 'Partner': 0, 'Dependents': 0,
    'PhoneService': 1, 'MultipleLines': 0, 'OnlineBackup': 0,
    'DeviceProtection': 0, 'StreamingTV': 0, 'StreamingMovies': 0,
    'PaperlessBilling': 1, 'PaymentMethod': 2,
}

rows = []
for p in profiles:
    row = {**defaults, **{k: v for k, v in p.items() if k != 'label'}}
    x = np.array([[row[f] for f in feature_names]])
    prob = model.predict_proba(scaler.transform(x))[0, 1]
    rows.append({
        'Profile': p['label'],
        'Churn Prob': f'{prob:.3f}',
        'Prediction': 'CHURN' if prob >= 0.5 else 'STAY',
        'Risk': 'HIGH' if prob >= 0.7 else 'MEDIUM' if prob >= 0.45 else 'LOW'
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))